In [1]:
# 1. Instalação e Inicialização do PySpark
!pip install pyspark -q

from pyspark.sql import SparkSession
import pyspark.sql.functions as F

spark = SparkSession.builder \
    .appName("Sistematizacao_KDD_Diabetes") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()
print("Sessão Spark iniciada com sucesso!")

Sessão Spark iniciada com sucesso!


In [2]:
# 2. Download do dataset diretamente para o ambiente Colab
!pip install kagglehub -q
import kagglehub, shutil

path = kagglehub.dataset_download('alexteboul/diabetes-health-indicators-dataset')
shutil.copy(f'{path}/diabetes_binary_health_indicators_BRFSS2015.csv', './diabetes_binary_health_indicators_BRFSS2015.csv')
print("Dataset baixado com sucesso!")


Using Colab cache for faster access to the 'diabetes-health-indicators-dataset' dataset.
Dataset baixado com sucesso!


In [3]:
# 3. Leitura e limpeza (Remoção de duplicadas/nulos)
df = spark.read.csv(
    "diabetes_binary_health_indicators_BRFSS2015.csv",
    header=True,
    inferSchema=True
)

total_bruto = df.count()
df_limpo = df.dropDuplicates() .na.drop()
total_limpo = df_limpo.count()

print(f"Total bruto: {total_bruto}")
print(f"Total limpo: {total_limpo} (Duplicadas removidas): {total_bruto - total_limpo}")

# 4. Engenharia de atributos e criação de view temporária para SQL
df_transformado = df_limpo.withColumn("Risco_Cardiovascular", F.col("HighBP") + F.col("HighChol"))
df_transformado.createOrReplaceTempView("tabela_diabetes")
print("Tabela SQL 'tabela_diabetes' e variável 'df_transformado' prontas!")

Total bruto: 253680
Total limpo: 229474 (Duplicadas removidas): 24206
Tabela SQL 'tabela_diabetes' e variável 'df_transformado' prontas!


In [4]:
# Pergunta 1: Prevalência geral de Diabetes na amostra
print("=== 1. Proporção de Doença na Amostra ===")
spark.sql("""
 SELECT
    Diabetes_binary AS Diagnostico,
    COUNT(*) AS Total,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) AS Percentual
  FROM tabela_diabetes
  GROUP BY Diabetes_binary
""").show()

# Pergunta 2: IMC Médio por Diagnóstico
print("=== 2. Média e Dispersão do IMC ===")
spark.sql ("""
 SELECT
    Diabetes_binary AS Diagnostico,
    ROUND(AVG(BMI), 2) AS Media_IMC,
    ROUND(STDDEV(BMI), 2) AS Desvio_IMC
  FROM tabela_diabetes
  GROUP BY Diabetes_binary
""").show()

# Pergunta 3: Risco Cardiovascular Combinado (Hipertensão + Colesterol)
print("=== 3. Prevalência por Risco Cardiovascular (0=Nenhum, 1= Um deles, 2= Ambos) ===")
spark.sql("""
 SELECT
    Risco_Cardiovascular,
    COUNT(*) AS Total_Populacao,
    SUM(Diabetes_binary) AS Total_Diabeticos,
    ROUND(SUM(Diabetes_binary) * 100.0 / COUNT(*), 2) AS Taxa_Diabetes_Pct
  FROM tabela_diabetes
  GROUP BY Risco_Cardiovascular
  ORDER BY Risco_Cardiovascular
""") .show()

# Pergunta 4: Prevalência de Diabetes por Faixa Etária
print("=== 4. Prevalência de Diabetes Idade (1=Jovem até 13=Idoso) ===")
spark.sql("""
 SELECT
    AGE AS Faixa_Idade,
    COUNT(*) AS Total,
    ROUND(SUM(Diabetes_binary) * 100.0 / COUNT(*), 2) AS Taxa_Diabetes_Pct
  FROM tabela_diabetes
  GROUP BY Age
  ORDER BY Age
""").show(14)

# Pergunta 5: Efeito de Hábitos Saudáveis
print("=== 5. Hábitos Saudáveis x Diabetes ===")
spark.sql("""
 SELECT
    PhysActivity AS Atividade_Fisica,
    Veggies AS Come_Vegetais,
    COUNT(*) AS Total,
    ROUND(SUM(Diabetes_binary) * 100.0 / COUNT(*), 2) AS Taxa_Diabetes_Pct
  FROM tabela_diabetes
  GROUP BY PhysActivity, Veggies
  ORDER BY Taxa_Diabetes_Pct DESC
""").show()

=== 1. Proporção de Doença na Amostra ===
+-----------+------+----------+
|Diagnostico| Total|Percentual|
+-----------+------+----------+
|        0.0|194377|     84.71|
|        1.0| 35097|     15.29|
+-----------+------+----------+

=== 2. Média e Dispersão do IMC ===
+-----------+---------+----------+
|Diagnostico|Media_IMC|Desvio_IMC|
+-----------+---------+----------+
|        0.0|     28.1|       6.5|
|        1.0|    31.96|      7.38|
+-----------+---------+----------+

=== 3. Prevalência por Risco Cardiovascular (0=Nenhum, 1= Um deles, 2= Ambos) ===
+--------------------+---------------+----------------+-----------------+
|Risco_Cardiovascular|Total_Populacao|Total_Diabeticos|Taxa_Diabetes_Pct|
+--------------------+---------------+----------------+-----------------+
|                 0.0|          86026|          4246.0|             4.94|
|                 1.0|          81291|         11801.0|            14.52|
|                 2.0|          62157|         19050.0|           